## The Price Is Right - Building Advanced RAG with ChromaDB & Vector Store

The RAG(Retrieval Augmented Generation) based on a dataset of 800,000 scraped Amazon Products.

    - We discovered that LLMs are really good at this

    - We are going to try inference time techniques instead of training -- by using RAG!

In [1]:
# libraries
import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm
from agents.evaluator import evaluate
from agents.items import Item

In [3]:
# setting the environment
load_dotenv(override=True)
DB = 'products_vectorstore'

hf_token = os.environ['HF_TOKEN']
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
# loading the datasets
LITE_MODE = False

username = 'ed-donner'
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training, {len(val):,} validation items, & {len(test):,} test items")

Loaded 800,000 training, 10,000 validation items, & 10,000 test items


### Creating the Chroma Datastore

In [6]:
client = chromadb.PersistentClient(path=DB)

### Introducing the SentenceTransformer Encoding LLM

In [7]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
# example
vector = encoder.encode(["A proficient AI engineer who has almost reached the finale of AI Engineering Core Track!"])[0]
print(vector.shape)
vector

(384,)


array([-5.68233617e-02, -6.70465529e-02,  4.41130176e-02,  5.98603580e-03,
       -2.28948928e-02, -2.95300856e-02,  5.56369722e-02,  3.42665762e-02,
       -1.08529426e-01, -3.81690711e-02, -7.43872300e-02, -1.03664264e-01,
        1.69148091e-02,  1.33924768e-03, -6.86191097e-02,  8.99353549e-02,
       -1.45186437e-02, -2.43885629e-02,  4.21848521e-03, -9.62992534e-02,
       -2.51799524e-02,  4.60676216e-02,  4.95346682e-03, -3.88679318e-02,
        1.07717537e-03,  6.82337508e-02, -1.13859801e-02, -5.83416149e-02,
       -1.03801209e-02, -1.74952969e-02, -1.86478980e-02,  4.07059537e-03,
        1.59437954e-02,  6.49722293e-02,  3.71175483e-02,  2.78225299e-02,
       -4.41944860e-02, -2.34373212e-02,  9.71035436e-02, -5.06138876e-02,
       -1.93864070e-02, -3.83472070e-02,  4.76066843e-02, -3.36106792e-02,
        5.08287326e-02,  3.57935354e-02,  2.91817472e-03, -1.06529169e-01,
        4.07211781e-02, -5.85431349e-04, -1.05607428e-01, -1.03584334e-01,
        3.71124223e-02, -

In [10]:
collection_name = 'products'
existing_collection_names = [collection.name for collection in client.list_collections()]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)

    for i in tqdm(range(0, len(train), 1000)):
        documents = [item.summary for item in train[i: i+1000]]

        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
        ids = [f"doc_{j}" for j in range(i, i+1000)]
        ids = ids[:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

  0%|          | 0/800 [00:00<?, ?it/s]

In [11]:
MAXIMUM_DATAPOINTS = 10_000
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [12]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [13]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [14]:
# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vectorstore Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [15]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [16]:
# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [17]:
test[0]

<Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal = $219.0>

In [18]:
def vector(item):
    return encoder.encode(item.summary)

In [19]:
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results['documents'][0][:]
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents, prices

In [20]:
find_similars(test[0])

(['Title: Old Blood Noise Endeavors Procession Reverb  \nCategory: Audio Effects  \nBrand: Old Blood Noise Endeavors  \nDescription: A compact, sci‑fi inspired reverb pedal with three modulation modes for creating otherworldly echo effects.  \nDetails: Features adjustable mix, decay, speed, depth, and footswitches for bypass and hold; powered by 9\u202fV DC with 60\u202fmA draw.',
  'Title: Boss MD‑2 Mega Distortion Modulation Multi‑Effects Pedal  \nCategory: Music Equipment → Effects Pedals  \nBrand: Boss  \nDescription: A powerful distortion pedal that delivers extreme low‑end crunch and endless sustain for metal and hard rock.  \nDetails: Features a Gain Boost circuit, bottom‑heavy Bottom control for 6/7‑string guitars, adjustable Tone knob, and 1\u202fMΩ input impedance.',
  'Title: Old Blood Noise Endeavors Mondegreen Delay Pedal  \nCategory: Musical Instruments / Effects Pedals  \nBrand: Old Blood Noise  \nDescription: A digital delay pedal that transforms your signal into creati

In [21]:
def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"

    for similar, price in zip(similars, prices):
        message += f"Potentially related product: \n{similar}\nPrice is ${price:.2f}\n\n"

    return message

In [22]:
documents, prices = find_similars(test[0])
print(make_context(documents, prices))

For context, here are some other items that might be similar to the item you need to estimate.

Potentially related product: 
Title: Old Blood Noise Endeavors Procession Reverb  
Category: Audio Effects  
Brand: Old Blood Noise Endeavors  
Description: A compact, sci‑fi inspired reverb pedal with three modulation modes for creating otherworldly echo effects.  
Details: Features adjustable mix, decay, speed, depth, and footswitches for bypass and hold; powered by 9 V DC with 60 mA draw.
Price is $209.00

Potentially related product: 
Title: Boss MD‑2 Mega Distortion Modulation Multi‑Effects Pedal  
Category: Music Equipment → Effects Pedals  
Brand: Boss  
Description: A powerful distortion pedal that delivers extreme low‑end crunch and endless sustain for metal and hard rock.  
Details: Features a Gain Boost circuit, bottom‑heavy Bottom control for 6/7‑string guitars, adjustable Tone knob, and 1 MΩ input impedance.
Price is $109.99

Potentially related product: 
Title: Old Blood Noise 

In [23]:
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [24]:
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices))

[{'role': 'user', 'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.\n\nFor context, here are some other items that might be similar to the item you need to estimate.\n\nPotentially related product: \nTitle: Old Blood Noise Endeavors Procession Reverb  \nCategory: Audio Effects  \nBrand: Old Blood Noise Endeavors  \nDescription: A compact, sci‑fi inspired reverb pedal with three modulation modes for creating otherworldly echo effects.  \nDetails: Features adjustable mix, decay, speed, depth, and foo

In [25]:
def gpt_5_1_reg(item):
    documents, prices = find_similars(item)
    response = completion(model='gpt-5.1', messages=messages_for(item, documents, prices), reasoning_effort='none', seed=42)
    return response.choices[0].message.content

In [26]:
test[0].price

219.0

In [27]:
gpt_5_1_reg(test[0])

/Users/irenebusah/Desktop/ai-engineer-4-ds/lessons/llm-engineering/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:1463: UserWarning: Item 'summary' on TypedDict class 'ChatCompletionReasoningItem' is using the `ReadOnly` qualifier. Pydantic will not protect items from any mutation on dictionary instances.
  warnings.warn(


'$229.00'

In [28]:
evaluate(gpt_5_1_reg, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $24 $20 $23 $30 $130 $21 $5 $8 $40 $43 $50 $0 $7 $10 $3 $11 $30 $43 $10 $49 $11 $15 $15 $52 $193 $165 $3 $106 $52 $4 $5 $0 $40 $5 $70 $55 $31 $44 $12 $60 $10 $0 $5 $60 $5 $25 $0 $80 $10 $6 $20 $163 $4 $27 $46 $33 $39 $93 $3 $117 $41 $4 $60 $199 $10 $70 $256 $0 $19 $13 $8 $0 $4 $25 $13 $11 $3 $1 $5 $20 $3 $0 $35 $12 $5 $53 $11 $50 $8 $11 $10 $3 $5 $0 $78 $6 $52 $60 $155 $0 $13 $2 $10 $15 $8 $5 $265 $5 $114 $7 $13 $1 $40 $24 $0 $9 $5 $64 $86 $1 $11 $1 $4 $0 $64 $6 $21 $10 $19 $0 $23 $10 $1 $65 $0 $85 $15 $47 $12 $24 $49 $15 $10 $5 $11 $5 $75 $5 $5 $4 $16 $2 $10 $0 $11 $36 $2 $10 $0 $91 $12 $2 $2 $191 $3 $141 $15 $0 $2 $24 $3 $90 $17 $28 $21 $1 $30 $26 $9 $59 $15 $150 $30 $20 $17 $33 $14 $18 $2 $13 $8 $10 $26 $0 $0 $25 $12 $3 $6 

In [29]:
import modal

In [30]:
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()

In [31]:
def specialist(item):
    return pricer.price.remote(item.summary)

In [32]:
def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

In [34]:
from agents.deep_neural_network import DeepNeuralNetworkInference

runner = DeepNeuralNetworkInference()
runner.setup()
runner.load("deep_neural_network.pth")

In [35]:
def deep_neural_network(item):
    return runner.inference(item.summary)

In [36]:
def ensemble(item):
    price1 = get_price(gpt_5_1_reg(item))
    price2 = specialist(item)
    price3 = deep_neural_network(item)

    return price1 * 0.8 + price2 * 0.1 + price3 * 0.1

In [39]:
evaluate(ensemble, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$14 $32 $17 $15 $26 $130 $6 $6 $7 $42 $48 $53 $1 $6 $8 $2 $15 $28 $37 $39 $34 $10 $14 $28 $61 $192 $194 $0 $114 $52 $2 $6 $8 $27 $6 $86 $49 $30 $46 $8 $87 $16 $1 $12 $70 $6 $21 $1 $82 $6 $7 $19 $163 $10 $29 $22 $22 $38 $88 $6 $126 $41 $2 $56 $236 $12 $60 $270 $0 $24 $13 $8 $12 $7 $22 $12 $10 $3 $2 $5 $6 $6 $1 $40 $11 $1 $46 $13 $34 $4 $9 $7 $3 $4 $1 $68 $5 $41 $50 $167 $1 $5 $1 $17 $11 $13 $2 $266 $6 $101 $10 $13 $0 $41 $17 $5 $2 $6 $60 $83 $1 $59 $2 $11 $4 $59 $5 $21 $2 $26 $4 $20 $9 $1 $62 $0 $77 $11 $50 $12 $21 $27 $16 $13 $5 $25 $3 $7 $11 $6 $3 $23 $3 $1 $1 $12 $41 $2 $27 $2 $70 $15 $4 $2 $210 $1 $210 $16 $1 $4 $27 $2 $130 $4 $28 $17 $0 $27 $14 $9 $43 $12 $114 $23 $17 $16 $42 $9 $18 $0 $11 $5 $5 $24 $1 $6 $25 $8 $6 $5 

In [40]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [41]:
from agents.frontier_agent import FrontierAgent

agent = FrontierAgent(collection)
agent.price("Quadcast HperX condenser mic, connects via usb-c to your computer for crystal clear audio")

INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.base.model:No device provided, using mps
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $109.99


109.99

In [42]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $289.00


289.0

In [43]:
from agents.neural_network_agent import NeuralNetworkAgent

agent = NeuralNetworkAgent()
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using mps
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $162.39


162.3922576904297

In [44]:
from agents.ensemble_agent import EnsembleAgent

agent = EnsembleAgent(collection)

INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.base.model:No device provided, using mps
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using mps
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready


In [45]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text


00:19:10 - LiteLLM:INFO: utils.py:4502 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


00:19:12 - LiteLLM:INFO: utils.py:1864 - Wrapper: Completed Call, calling success_handler


INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $299.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $289.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $187.49
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $279.85


279.8489410400391